# Module 14 — Audit log and the human gate

**THE ONE IDEA:** **you cannot retrofit an audit trail.** If the trace was not written
while the agent ran, the information is gone — and a regulated decision you cannot
reconstruct is a decision you did not legitimately make.

Two mechanisms, both non-negotiable in a bank:

1. **Append-only JSONL trace** — every call, its arguments, its result, its latency
2. **A human gate on write tools** — `_tools.py` marks `confirm_decision` as a
   `WRITE_TOOL`; it has a side effect, so a person approves it before it runs

Reads are free. **Writes need a name against them.**


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json, time, pathlib
from _fake_model import FakeModel, tool_turn, text_turn
from _tools import run_tool, WRITE_TOOLS

TRACE = pathlib.Path("run_trace.jsonl")
TRACE.unlink(missing_ok=True)

def emit(event, **fields):
    """Append-only. One JSON object per line, flushed immediately.
    Never rewrite a line — an audit log you can edit is not an audit log."""
    rec = {"ts": round(time.time(), 3), "event": event, **fields}
    with TRACE.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    return rec

print("write tools needing a human gate:", WRITE_TOOLS)

## The approval gate

`approve_fn` is injected so the notebook can run unattended. In production this is a
queue, a Slack button, or a case-management screen — the shape does not change.

In [ ]:
def traced_agent(fake, approve_fn, max_steps=6, run_id="run-001"):
    emit("run_start", run_id=run_id)
    messages = [{"role": "user", "content": "Approve the ERC waiver for case C-1002."}]
    for step in range(1, max_steps + 1):
        t0 = time.time()
        r = fake.create(messages=messages)
        msg = r.choices[0].message
        emit("llm_turn", run_id=run_id, step=step, finish=r.choices[0].finish_reason,
             tokens_in=r.usage.prompt_tokens, tokens_out=r.usage.completion_tokens,
             latency_ms=round((time.time() - t0) * 1000))

        if r.choices[0].finish_reason != "tool_calls":
            emit("run_end", run_id=run_id, outcome="answered", steps=step)
            return msg.content

        for tc in msg.tool_calls:
            name, args = tc.function.name, json.loads(tc.function.arguments)

            if name in WRITE_TOOLS:                       # ── THE HUMAN GATE ──
                emit("approval_requested", run_id=run_id, step=step, tool=name, args=args)
                ok, who = approve_fn(name, args)
                emit("approval_decision", run_id=run_id, step=step, tool=name,
                     approved=ok, approver=who)
                if not ok:
                    out = "DENIED: a human declined this action."
                    emit("tool_blocked", run_id=run_id, step=step, tool=name)
                    messages.append({"role": "user", "content": out}); continue

            t1 = time.time(); out = run_tool(name, args)
            emit("tool_call", run_id=run_id, step=step, tool=name, args=args,
                 result=out[:120], latency_ms=round((time.time() - t1) * 1000),
                 write=name in WRITE_TOOLS)
            messages.append({"role": "user", "content": out})
    emit("run_end", run_id=run_id, outcome="capped", steps=max_steps)
    return "ABORT: iteration cap"

SCRIPT = [tool_turn("fetch_customer_note", {"customer_id": "C-1002"}, "c1"),
          tool_turn("confirm_decision", {"reference": "ERC-WAIVER-1002"}, "c2"),
          text_turn("Waiver recorded.")]

## Run 1 — the human approves

In [ ]:
print(traced_agent(FakeModel(list(SCRIPT)), lambda n, a: (True, "s.khan"), run_id="run-approve"))

## Run 2 — the human declines

The read tool still runs. Only the **write** is blocked, and the refusal goes back to the
model as an observation rather than an exception.

In [ ]:
print(traced_agent(FakeModel(list(SCRIPT)), lambda n, a: (False, "s.khan"), run_id="run-deny"))

## Reading the trace back

In [ ]:
events = [json.loads(l) for l in TRACE.read_text().splitlines()]
print(f"{'run':12} {'event':20} {'detail':40}")
print("-" * 74)
for e in events:
    detail = (e.get("tool") or e.get("outcome") or f"step {e.get('step','')}")
    if e["event"] == "approval_decision":
        detail = f"{e['tool']} approved={e['approved']} by {e['approver']}"
    print(f"{e.get('run_id',''):12} {e['event']:20} {str(detail)[:40]:40}")

writes = [e for e in events if e["event"] == "tool_call" and e.get("write")]
gates  = [e for e in events if e["event"] == "approval_decision"]
print(f"\n{len(events)} events · {len(writes)} write(s) executed · {len(gates)} gate decision(s)")
print()
print("LESSON — every write that ran has an approval_decision BEFORE it, naming a")
print("person. The denied run has the request, the refusal, and a tool_blocked —")
print("and no COMMITTED line anywhere. That is what 'reconstructable' means.")
print()
print("Three properties that make this an audit log rather than logging:")
print("  1. APPEND-ONLY — a line is never rewritten. Editable history is not evidence.")
print("  2. Written DURING the run, not after. You cannot reconstruct latency, or")
print("     arguments the model later discarded, from a final answer.")
print("  3. The gate is on the TOOL REGISTRY (WRITE_TOOLS), not on the prompt. A")
print("     model cannot talk its way past a Python `if`.")
print()
print("This is the SS1/23 model-risk traceability requirement and the Consumer Duty")
print("approval requirement, in about forty lines. Module 17 widens the same trace")
print("into full observability: cost, tokens and latency per run.")

---

**Next:** Block E — `../E_security/15_prompt_injection_via_tools.ipynb`, where an
attacker uses a tool result to try to reach that write tool.